# 🧑‍💻 Ejercicios: Palantir
# Plantilla: salarios, niveles y stock options

⚠️ Datos SIMULADOS / ilustrativos
(generados con pandas + numpy).
No son cifras oficiales de Palantir.

### 💼 PREGUNTA ENTREVISTA:
### ¿Qué departamento
### cuesta más en
### nóminas totales?

Headcount x salario medio = coste.
No es lo mismo caro que numeroso.

In [ ]:
import pandas as pd

df = pd.read_csv(
  '../datasets/palantir_empleados.csv'
)

r = (
  df
  .groupby('departamento')
  .agg(
    headcount=('salario_usd',
               'size'),
    medio=('salario_usd', 'mean'),
    total=('salario_usd', 'sum'),
  )
  .round(0)
  .sort_values('total',
    ascending=False)
)

print('Coste nóminas por depto:')
print(r)

### 💼 PREGUNTA ENTREVISTA:
### ¿Cuánto sube el
### salario de Junior
### a Principal? (%)

Mediana por nivel y el salto
acumulado respecto a Junior.

In [ ]:
import pandas as pd

df = pd.read_csv(
  '../datasets/palantir_empleados.csv'
)

orden = ['Junior', 'Mid', 'Senior',
         'Staff', 'Principal']

r = (
  df
  .groupby('nivel')['salario_usd']
  .median()
  .reindex(orden)
)

r = r.to_frame('mediana')
r['vs_junior_%'] = (
  r['mediana'] / r['mediana'].iloc[0]
  - 1
) * 100

print('Escala salarial por nivel:')
print(r.round(0))

### 💼 PREGUNTA ENTREVISTA:
### ¿Las stock options
### crecen con la
### antigüedad? (corr)

Correlación y tramos de antigüedad
para ver el efecto fidelización.

In [ ]:
import pandas as pd

df = pd.read_csv(
  '../datasets/palantir_empleados.csv'
)

corr = (
  df['antiguedad']
  .corr(df['stock_options'])
)

df['tramo'] = pd.cut(
  df['antiguedad'],
  bins=[0, 2, 5, 100],
  labels=['0-2a', '2-5a', '5a+'],
)

r = (
  df
  .groupby('tramo', observed=True)
  ['stock_options']
  .mean()
  .round(0)
)

print(f'Correlación: {corr:.2f}')
print('\nStock medio por tramo:')
print(r)

### 💼 PREGUNTA ENTREVISTA:
### ¿Qué ciudad paga
### mejor a mismo
### nivel? (justo)

Comparar ciudades sin sesgo: fijamos
nivel Senior y vemos el salario.

In [ ]:
import pandas as pd

df = pd.read_csv(
  '../datasets/palantir_empleados.csv'
)

r = (
  df[df['nivel'] == 'Senior']
  .groupby('ciudad')['salario_usd']
  .median()
  .round(0)
  .sort_values(ascending=False)
)

base = r.min()
dif = (r / base - 1) * 100

print('Senior - salario por ciudad:')
print(r)
print('\nSobrecoste vs mínima (%):')
print(dif.round(1))

### 💼 PREGUNTA ENTREVISTA:
### ¿Qué peso tiene la
### cúpula (Staff+) en
### el coste total?

Pocos perfiles senior pueden
concentrar gran parte del gasto.

In [ ]:
import pandas as pd

df = pd.read_csv(
  '../datasets/palantir_empleados.csv'
)

top = ['Staff', 'Principal']
df['cupula'] = df['nivel'].isin(top)

r = (
  df
  .groupby('cupula')
  .agg(
    pers=('salario_usd', 'size'),
    coste=('salario_usd', 'sum'),
  )
)

r['%_pers'] = (
  r['pers'] / r['pers'].sum() * 100
).round(1)
r['%_coste'] = (
  r['coste'] / r['coste'].sum() * 100
).round(1)

print('Cúpula (True) vs resto:')
print(r[['%_pers', '%_coste']])